# Módulo 18 · Práctica: mezcla de expertos y KV cache con GQA

En este notebook construyes en PyTorch una **capa de mezcla de expertos (MoE)** con 4 expertos y enrutado top-2, la metes en un bloque transformer pequeño y la comparas con un MLP denso. Después calculas cuánta memoria ahorra la atención **GQA** en la KV cache. Todo corre en CPU en menos de tres minutos.

**Cómo trabajar:** ejecuta en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que las sigue.

| Parte | Qué haces |
| --- | --- |
| 0 | Preparación y datos sintéticos «multilingües» |
| 1 | El router top-k |
| 2 | La capa MoE y la pérdida de balanceo |
| 3 | Bloque transformer: MoE frente a MLP denso |
| 4 | KV cache: MHA, GQA y MQA |
| 5 | Retos |

## Parte 0 · Preparación y datos

El problema sintético imita un corpus con **4 «idiomas»**. Cada secuencia empieza con un marcador de idioma; después, cada token se obtiene del anterior con una permutación propia de ese idioma (con un 10 % de ruido). Un buen modelo debe leer el marcador y aplicar la regla correcta: un trabajo natural para expertos especializados.

In [ ]:
import math, time
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
inicio = time.time()

NV, NL, T = 32, 4, 16          # tokens normales, idiomas, longitud
V = NV + NL                    # vocabulario: 32 tokens + 4 marcadores
g = torch.Generator().manual_seed(1)
perms = torch.stack([torch.randperm(NV, generator=g) for _ in range(NL)])

def lote(B, gen=None):
    l = torch.randint(0, NL, (B,), generator=gen)
    cur = torch.randint(0, NV, (B,), generator=gen)
    seq = [NV + l, cur]
    for _ in range(T - 1):
        cur = perms[l, cur]
        ruido = torch.rand(B, generator=gen) < 0.1
        cur = torch.where(ruido, torch.randint(0, NV, (B,), generator=gen), cur)
        seq.append(cur)
    s = torch.stack(seq, 1)
    return s[:, :-1], s[:, 1:]          # entrada y objetivo desplazado

x, y = lote(1)
print("x:", x[0].tolist()); print("y:", y[0].tolist())
# Pérdida mínima alcanzable (entropía del ruido): ningún modelo puede bajar de aquí
p_ok, p_mal = 0.9 + 0.1 / NV, 0.1 / NV
suelo = -(p_ok * math.log(p_ok) + (NV - 1) * p_mal * math.log(p_mal))
print(f"Suelo teórico: {suelo:.4f} nats   ·   uniforme: {math.log(NV):.4f} nats")

## Parte 1 · El router top-k

El router es una capa lineal que da un logit por experto. Nos quedamos con los $k$ mayores y normalizamos con softmax **solo entre ellos** (como Mixtral):

$$\text{pesos} = \operatorname{softmax}(\text{TopK}(x W_r)), \qquad y = \sum_{i \in \text{TopK}} \text{peso}_i \, E_i(x)$$

**Ejercicio 1.** Implementa `enrutar(logits, k)` con `topk` y `F.softmax`.

In [ ]:
# TU CÓDIGO
def enrutar(logits, k):
    """logits: (N, E). Devuelve pesos (N, k) que suman 1 por fila e índices (N, k)."""
    pesos, indices = None, None
    return pesos, indices

In [ ]:
# Comprobación
lg = torch.tensor([[2.0, 1.0, 0.5, -1.0],
                   [0.5, 2.0, 1.2, 0.0]])
pesos, idx = enrutar(lg, 2)
assert pesos is not None, "enrutar devuelve None"
assert idx.tolist() == [[0, 1], [1, 2]], f"Índices incorrectos: {idx.tolist()}"
assert torch.allclose(pesos.sum(-1), torch.ones(2)), "Los pesos de cada fila deben sumar 1"
assert torch.allclose(pesos[0], torch.tensor([0.7311, 0.2689]), atol=1e-4), "Fila 0: softmax de (2, 1)"
assert torch.allclose(pesos[1], torch.tensor([0.6900, 0.3100]), atol=1e-4), "Fila 1: softmax de (2, 1,2)"
print("Correcto:", pesos)

## Parte 2 · La capa MoE y la pérdida de balanceo

Sin control, el router tiende a mandar casi todo a uno o dos expertos. La pérdida auxiliar del Switch Transformer (2021) lo penaliza:

$$\mathcal{L}_{\text{bal}} = N \sum_{i=1}^{N} f_i \, P_i$$

$f_i$ = fracción de asignaciones que recibe el experto $i$ (con top-$k$, dividimos entre $k$ para que $\sum f_i = 1$); $P_i$ = probabilidad media que el router (softmax sobre los $N$ expertos) da al experto $i$. Vale 1 con reparto uniforme y $N$ si todo va a un experto.

**Ejercicio 2.** Implementa `perdida_balanceo`. Pista: `F.one_hot(indices, n_exp)` tiene forma (tokens, k, n_exp).

In [ ]:
# TU CÓDIGO
def perdida_balanceo(logits, indices, n_exp):
    """L_bal = N · Σ_i f_i · P_i (Switch Transformer, generalizada a top-k)."""
    f = None   # fracción de asignaciones que recibe cada experto, forma (n_exp,)
    P = None   # probabilidad media del router para cada experto, forma (n_exp,)
    perdida = None
    return perdida, f

In [ ]:
# Comprobación
N = 4
uniforme = torch.zeros(8, N)                                 # router indiferente
idx_u = torch.tensor([[0, 1], [2, 3]] * 4)                   # reparto perfecto
l_u, f_u = perdida_balanceo(uniforme, idx_u, N)
assert l_u is not None, "perdida_balanceo devuelve None"
assert torch.allclose(f_u, torch.full((N,), 0.25)), f"f debería ser uniforme: {f_u}"
assert abs(l_u.item() - 1.0) < 1e-5, f"Reparto uniforme → 1, no {l_u.item():.4f}"
colapso = torch.tensor([[50.0, 40.0, 0.0, 0.0]] * 8)
l_c, _ = perdida_balanceo(colapso, torch.tensor([[0, 1]] * 8), N)
assert abs(l_c.item() - 2.0) < 1e-3, f"Mitad a 0 y mitad a 1 con P≈(1,0,0,0) → 2, no {l_c.item():.4f}"
print("Correcto: uniforme", round(l_u.item(), 4), "· colapsado", round(l_c.item(), 4))

Con `enrutar` y `perdida_balanceo` montamos la capa. Cada experto es un MLP pequeño. El bucle recorre los expertos y procesa solo los tokens que les tocan: así el cómputo por token es el de 2 expertos, no el de 4.

In [ ]:
class Experto(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.w1 = nn.Linear(d, h); self.w2 = nn.Linear(h, d)
    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))

class MoE(nn.Module):
    def __init__(self, d, h, n_exp=4, k=2):
        super().__init__()
        self.router = nn.Linear(d, n_exp, bias=False)
        self.expertos = nn.ModuleList([Experto(d, h) for _ in range(n_exp)])
        self.n_exp, self.k = n_exp, k
    def forward(self, x):
        B, T_, D = x.shape
        xf = x.reshape(-1, D)
        logits = self.router(xf)
        pesos, idx = enrutar(logits, self.k)
        y = torch.zeros_like(xf)
        for e, experto in enumerate(self.expertos):
            fila, pos = (idx == e).nonzero(as_tuple=True)   # tokens enviados a e
            if fila.numel() > 0:
                y.index_add_(0, fila, pesos[fila, pos, None] * experto(xf[fila]))
        self.aux, self.uso = perdida_balanceo(logits, idx, self.n_exp)
        return y.reshape(B, T_, D)

class Denso(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.mlp = Experto(d, h); self.aux = torch.tensor(0.0)
    def forward(self, x):
        return self.mlp(x)

capa = MoE(8, 16)
salida = capa(torch.randn(2, 5, 8))
print(salida.shape, "· aux =", round(capa.aux.item(), 3), "· uso =", capa.uso)

In [ ]:
# Comprobación
assert salida.shape == (2, 5, 8), "La MoE debe conservar la forma"
assert 1.0 <= capa.aux.item() <= 4.0, "La pérdida de balanceo debe estar entre 1 y N"
assert abs(capa.uso.sum().item() - 1) < 1e-5, "Las fracciones f deben sumar 1"
print("Correcto")

## Parte 3 · Bloque transformer: MoE frente a MLP denso

Un bloque pre-norm (módulo 14): atención causal + FFN, ambas con conexión residual. Solo cambiamos la FFN. Comparamos tres modelos:

| Modelo | FFN |
| --- | --- |
| Denso h=32 | Un MLP 32→32→32 (mismo cómputo por token que **un** experto) |
| MoE 4×32, top-2 | 4 expertos 32→32→32, se usan 2 por token |
| Denso h=64 | Un MLP 32→64→32 (mismo cómputo por token que la MoE) |

In [ ]:
class Bloque(nn.Module):
    def __init__(self, d, n_head, ffn, T_max):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.att = nn.MultiheadAttention(d, n_head, batch_first=True)
        self.ffn = ffn
        self.register_buffer("mask", torch.triu(torch.ones(T_max, T_max, dtype=torch.bool), 1))
    def forward(self, x):
        T_ = x.shape[1]; h = self.ln1(x)
        a, _ = self.att(h, h, h, attn_mask=self.mask[:T_, :T_], need_weights=False)
        x = x + a
        return x + self.ffn(self.ln2(x))

class MiniLM(nn.Module):
    def __init__(self, ffn, d=32):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, d), nn.Embedding(T, d)
        self.bloque = Bloque(d, 2, ffn, T)
        self.ln, self.out = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.shape[1]))
        return self.out(self.ln(self.bloque(h)))

def perdida(modelo, x, y):
    logits = modelo(x)[:, 1:]               # ignoramos la predicción tras el marcador
    return F.cross_entropy(logits.reshape(-1, V), y[:, 1:].reshape(-1))

def entrenar(modelo, pasos=300, lam=0.01):
    torch.manual_seed(0)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-3)
    for _ in range(pasos):
        x, y = lote(64)
        loss = perdida(modelo, x, y) + lam * modelo.bloque.ffn.aux
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x, y = lote(2048, torch.Generator().manual_seed(123))
        return perdida(modelo, x, y).item()

contar = lambda m: sum(p.numel() for p in m.parameters())

In [ ]:
resultados = {}
for nombre, hacer in [("Denso h=32", lambda: Denso(32, 32)),
                      ("MoE 4x32 top-2", lambda: MoE(32, 32)),
                      ("Denso h=64", lambda: Denso(32, 64))]:
    torch.manual_seed(0)
    ffn = hacer(); modelo = MiniLM(ffn)
    t0 = time.process_time(); val = entrenar(modelo)
    resultados[nombre] = (val, contar(ffn), ffn)
    print(f"{nombre:15s} FFN={contar(ffn):5d} parám.  val={val:.4f}  ({time.process_time()-t0:.0f} s CPU)")
denso32_ffn = resultados["Denso h=32"][2]
moe_ffn = resultados["MoE 4x32 top-2"][2]
print("Uso de cada experto en el último lote:", moe_ffn.uso)

**Ejercicio 3.** Calcula cuántos parámetros de la FFN **se usan para un token** en la MoE (router + 2 expertos) y en el denso h=32. Usa `contar`.

In [ ]:
# TU CÓDIGO
# Parámetros de la capa FFN que se usan para procesar UN token
activos_moe = None      # router + 2 expertos
activos_denso32 = None

In [ ]:
# Comprobación
assert activos_moe == 4224 + 128, f"Router (128) + 2 expertos (2·2112): {activos_moe}"
assert activos_denso32 == 2112, f"Denso h=32: {activos_denso32}"
v = {k: r[0] for k, r in resultados.items()}
assert v["MoE 4x32 top-2"] < v["Denso h=32"], "La MoE debería superar al denso del mismo tamaño de experto"
assert moe_ffn.uso.min() > 0.1, "Algún experto apenas se usa: revisa la pérdida de balanceo"
print(f"Correcto. Totales MoE: {contar(moe_ffn)}; activos: {activos_moe} ({activos_moe/contar(moe_ffn):.0%})")

**Interpretación.** La MoE tiene unas 4 veces los parámetros del denso h=32 y usa unos 2 veces su cómputo; baja claramente la pérdida. Frente al denso h=64 (mismo cómputo activo) suele quedar algo mejor: más parámetros a igual cómputo. En modelos grandes ese es justo el atractivo de Mixtral o Grok-1.

## Parte 4 · KV cache: MHA, GQA y MQA

Al generar, cada capa guarda K y V de todos los tokens previos (módulo 19). Su tamaño:

$$\text{bytes} = 2 \cdot n_{\text{capas}} \cdot n_{kv} \cdot d_{\text{cabeza}} \cdot T \cdot \text{lote} \cdot \text{bytes por valor}$$

El 2 cuenta K y V. En MHA $n_{kv}$ = número de cabezas; en GQA es el número de grupos; en MQA, 1.

**Ejercicio 4.** Implementa `kv_cache_bytes` (bf16 → 2 bytes por defecto).

In [ ]:
# TU CÓDIGO
def kv_cache_bytes(n_capas, n_kv, d_cabeza, T, lote=1, bytes_por_valor=2):
    return None

In [ ]:
# Comprobación
GiB, MiB = 2**30, 2**20
# Llama 2 70B (artículo de 2023): 80 capas, 64 cabezas de query, 8 de KV, d_cabeza = 128
mha = kv_cache_bytes(80, 64, 128, 4096)
gqa = kv_cache_bytes(80, 8, 128, 4096)
mqa = kv_cache_bytes(80, 1, 128, 4096)
assert mha is not None, "kv_cache_bytes devuelve None"
assert mha == 10 * GiB, f"MHA debería ocupar 10 GiB, no {mha/GiB:.3f}"
assert gqa == 1.25 * GiB and mqa == 0.15625 * GiB, "Revisa GQA o MQA"
for n, b in [("MHA", mha), ("GQA-8", gqa), ("MQA", mqa)]:
    print(f"{n:6s} {b/GiB:7.3f} GiB por secuencia de 4.096 tokens")
print("Correcto")

En GQA cada grupo de cabezas de query comparte una cabeza K/V. En código basta con **repetir** las cabezas K/V antes de la atención normal.

**Ejercicio 5.** Implementa `expandir_kv` con `repeat_interleave`.

In [ ]:
# TU CÓDIGO
def expandir_kv(k, n_head):
    """k: (B, n_kv, T, dh) -> (B, n_head, T, dh) repitiendo cada cabeza KV."""
    return None

In [ ]:
# Comprobación
B_, H, H_kv, T_, dh = 1, 8, 2, 5, 4
q = torch.randn(B_, H, T_, dh)
k = torch.randn(B_, H_kv, T_, dh); v_ = torch.randn(B_, H_kv, T_, dh)
ke = expandir_kv(k, H)
assert ke is not None and ke.shape == (B_, H, T_, dh), "Forma esperada (1, 8, 5, 4)"
assert torch.equal(ke[:, 0], ke[:, 3]) and torch.equal(ke[:, 4], k[:, 1]), "Cabezas 0–3 usan KV 0; 4–7 usan KV 1"
out = F.scaled_dot_product_attention(q, ke, expandir_kv(v_, H), is_causal=True)
print("Salida GQA:", tuple(out.shape), "· KV guardada:", tuple(k.shape), "→ 4 veces menos memoria")
print("Correcto")

## Parte 5 · Retos

1. Entrena la MoE con `lam=0` y `lam=0.1`. ¿Cambian el uso de los expertos y la pérdida?
2. ¿Qué experto elige primero cada token según su idioma? ¿Hay especialización?
3. Pasa a top-1 (estilo Switch). ¿Cuánto empeora la pérdida y cuánto baja el cómputo?
4. Calcula la KV cache de Mixtral 8x7B (32 capas, 8 cabezas KV de 128) a 32.768 tokens frente a MHA de 32 cabezas.

In [ ]:
print(f"Tiempo total: {time.time() - inicio:.0f} s reales, {time.process_time():.0f} s de CPU")

## Soluciones de referencia

Intenta los ejercicios antes de mirar. Descomenta para usar.

In [ ]:
# # --- Ejercicio 1 ---
# def enrutar(logits, k):
#     valores, indices = logits.topk(k, dim=-1)   # los k logits mayores
#     pesos = F.softmax(valores, dim=-1)          # softmax solo sobre los elegidos
#     return pesos, indices
#
# # --- Ejercicio 2 ---
# def perdida_balanceo(logits, indices, n_exp):
#     k = indices.shape[1]
#     f = F.one_hot(indices, n_exp).float().sum(dim=1).mean(dim=0) / k
#     P = F.softmax(logits, dim=-1).mean(dim=0)
#     perdida = n_exp * (f * P).sum()
#     return perdida, f
#
# # --- Ejercicio 3 ---
# activos_moe = contar(moe_ffn.router) + 2 * contar(moe_ffn.expertos[0])
# activos_denso32 = contar(denso32_ffn)
#
# # --- Ejercicio 4 ---
# def kv_cache_bytes(n_capas, n_kv, d_cabeza, T, lote=1, bytes_por_valor=2):
#     return 2 * n_capas * n_kv * d_cabeza * T * lote * bytes_por_valor
#
# # --- Ejercicio 5 ---
# def expandir_kv(k, n_head):
#     grupo = n_head // k.shape[1]
#     return k.repeat_interleave(grupo, dim=1)
# --- Reto 4 ---
# kv_cache_bytes(32, 8, 128, 32768) / GiB   # 4,0 GiB (MHA: 16,0 GiB)